<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/VITPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [4]:
#Transformer Block
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()

    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,dropout=0.2,batch_first=True)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.2)
    )

    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)

  def forward(self,input):
    attention_output, _ = self.attention(input,input,input)
    x = self.layernorm1(input + x)
    x = self.mlp(x)
    return self.layernorm2(attention_output + x)

In [5]:
class PatchEmbedding(nn.Module):
  def __init__(self,embed_dim,patch_size) -> None:
    super().__init__()

    self.patchEmbed = nn.Conv2d(in_channels=3,out_channels=embed_dim,kernel_size=patch_size,stride=patch_size)

  def forward(self,patch):
    x = self.patchEmbed(patch) # becomes [Batch, 128, 14, 14]
    x = x.flatten(2) # it basically convert those 14,14 into 196 and keep batch and features as it is
    x = x.transpose(1,2)

    return x

In [6]:
class VisionTransformer(nn.Module):
  def __init__(self,embed_dim,patch_size,image_size,ff_dim,num_heads,num_layers,num_classes) -> None:
    super().__init__()
    self.patchs = PatchEmbedding(embed_dim=embed_dim,patch_size=patch_size)

    num_patchs = (image_size // patch_size) ** 2

    self.positionEmbed = nn.Parameter(torch.zeros(1,num_patchs,embed_dim)) # Positional Embedding

    self.transformer_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])

    self.output_layer = nn.Linear(embed_dim,num_classes)

  def forward(self,x):
    x = self.patchs(x)
    x = x + self.positionEmbed

    for transform_layer in self.transformer_layers:
      x = transform_layer(x)

  # GLOBAL AVERAGE POOLING (The Pro Exit Strategy)
    x = x.mean(dim=1)

    return self.output_layer(x)

In [7]:
#Training
import torch.optim as optim

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer(embed_dim=128,
                          patch_size=16,
                          image_size=128,
                          ff_dim=128,
                          num_heads=4,
                          num_layers=2,
                          num_classes=4).to(device)

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.0001)

In [ ]:
# History dictionary to store the "Diary" of the model
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

epochs = 15

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # The 5 Sacred Steps
        optimizer.zero_grad()                      # Clear memory
        outputs = model(images)                    # Forward
        loss = loss_fn(outputs, labels)          # Calculate Error
        loss.backward()                            # Backprop
        optimizer.step()                           # Update Weights

        # Track Training Metrics
        train_running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    # --- VALIDATION PHASE ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad(): # No learning here, just testing
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            v_loss = loss_fn(outputs, labels)

            # Track Validation Metrics
            val_running_loss += v_loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # 2. CALCULATE AVERAGES
    # Loss: Total sum / Number of batches
    # Accuracy: Correct hits / Total images
    ep_train_loss = train_running_loss / len(train_loader)
    ep_train_acc  = train_correct / train_total
    ep_val_loss   = val_running_loss / len(val_loader)
    ep_val_acc    = val_correct / val_total